#SILVER_TRANSFORMATION for bronze.orders

In [0]:
from pyspark.sql.functions import row_number, col, current_timestamp
from pyspark.sql.window import Window

In [0]:
df = spark.table("ecommerce_lakehouse.bronze.orders").select("*")

In [0]:
%sql
SELECT 
    COUNT(*) AS total_orders, 
    COUNT(DISTINCT `customer_id`) AS distinct_customers 
FROM 
    `ecommerce_lakehouse`.`bronze`.`orders`;

In [0]:
df.limit(5).display()

###Step 1,2: dedupe on PK, deterministic ordering

In [0]:
w = Window.partitionBy("order_id").orderBy(
    col("order_purchase_timestamp").desc(),
    col("_load_timestamp").desc()       
)

df = (
    df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
df.count()

In [0]:
df.limit(5).display()

### Step 3 — null-key handling

In [0]:


df = (
    df
    .filter(col("order_id").isNotNull())      # no PK → unusable, can't be joined or deduped → drop
    .filter(col("customer_id").isNotNull())   # no customer → breaks FactSales→DimCustomer join later → drop
)



In [0]:
df.select("order_id", "customer_id").display()

In [0]:
df.groupBy("order_status").count().display()

In [0]:
df.count()

###Step 4 — write to Silver

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.silver")

In [0]:

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", True)
      .saveAsTable("ecommerce_lakehouse.silver.orders")
)

In [0]:
print(f"silver.orders: {spark.table('ecommerce_lakehouse.silver.orders').count():,} rows")

In [0]:
print("bronze.orders:", spark.table("ecommerce_lakehouse.bronze.orders").count())